# Merchant Extraction & Matching Prototype

**Goal**: Extract clean merchant names from raw bank transaction text and match them to a growing list of known merchants.

**Data**: 5 CSVs from `sample_data/csv/` (~2700 transactions) — SLSP (Slovak bank, UTF-16) and Revolut (English, UTF-8).

In [1]:
# Cell 1 — Setup & Imports
import pandas as pd
import re
from pathlib import Path
from collections import Counter
from rapidfuzz import fuzz, process

DATA_DIR = Path('../sample_data/csv')
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 100)

In [ ]:
# Cell 2 — Load All CSVs

def parse_amount(s: str) -> float:
    """Parse amount string handling various formats: '1 000,50' or '-4\xa0000.00'."""
    if not isinstance(s, str):
        return 0.0
    # Remove non-breaking spaces and regular spaces used as thousands separators
    t = s.replace('\xa0', '').replace(' ', '')
    # Handle comma as decimal separator (European format)
    if ',' in t and '.' not in t:
        t = t.replace(',', '.')
    elif ',' in t and '.' in t:
        # e.g. "1.000,50" -> "1000.50"
        t = t.replace('.', '').replace(',', '.')
    return float(t)

def load_slsp(path: Path) -> pd.DataFrame:
    """Load SLSP CSV (UTF-16, auto-detect delimiter)."""
    text = path.read_text(encoding='utf-16')
    sep = ';' if ';' in text.split('\n')[0] else ','
    df = pd.read_csv(path, encoding='utf-16', sep=sep, dtype=str)
    return pd.DataFrame({
        'bank': 'SLSP',
        'partner': df['Partner'].fillna(''),
        'description': df.get('Popis transakcie', pd.Series([''] * len(df))).fillna(''),
        'amount': df['Suma'].apply(parse_amount),
        'date': pd.to_datetime(df['Dátum splatnosti'], format='%d.%m.%Y'),
        'category': df.get('Kategória', pd.Series([''] * len(df))).fillna(''),
        'location': df.get('Miesto použitia karty', pd.Series([''] * len(df))).fillna(''),
        'tx_type': df.get('Typ transakcie', pd.Series([''] * len(df))).fillna(''),
        'partner_iban': df.get('IBAN partnera', pd.Series([''] * len(df))).fillna(''),
        'own_iban': df.get('Vlastný IBAN', pd.Series([''] * len(df))).fillna(''),
    })

def load_revolut(path: Path) -> pd.DataFrame:
    """Load Revolut CSV (UTF-8, comma)."""
    df = pd.read_csv(path, dtype=str)
    return pd.DataFrame({
        'bank': 'Revolut',
        'partner': df['Description'].fillna(''),
        'description': '',
        'amount': df['Amount'].apply(parse_amount),
        'date': pd.to_datetime(df['Completed Date']),
        'category': df.get('Type', pd.Series([''] * len(df))).fillna(''),
        'location': '',
        'tx_type': df.get('Type', pd.Series([''] * len(df))).fillna(''),
        'partner_iban': '',
        'own_iban': '',
    })

# Load all files
slsp_files = sorted(DATA_DIR.glob('SLSP/*.csv'))
revolut_files = sorted(DATA_DIR.glob('Revolut/*.csv'))

frames = []
for f in slsp_files:
    df = load_slsp(f)
    df['source_file'] = f.name
    frames.append(df)
    print(f'SLSP {f.name}: {len(df)} rows')

for f in revolut_files:
    df = load_revolut(f)
    df['source_file'] = f.name
    frames.append(df)
    print(f'Revolut {f.name}: {len(df)} rows')

txns = pd.concat(frames, ignore_index=True)
print(f'\nTotal: {len(txns)} transactions')
print(f'Date range: {txns.date.min()} to {txns.date.max()}')
print(f'\nBank distribution:')
print(txns.bank.value_counts())
print(f'\nPartner IBAN present: {(txns.partner_iban != "").sum()} / {len(txns)}')
print(f'Sample SLSP partners: {txns[txns.bank=="SLSP"].partner.head(10).tolist()}')
print(f'Sample Revolut partners: {txns[txns.bank=="Revolut"].partner.head(10).tolist()}')

In [3]:
# Cell 3 — Text Cleaning & Normalization

# Patterns to strip
IBAN_RE = re.compile(r'[A-Z]{2}\d{2}\s?\d{4}\s?\d{4}\s?\d{4}\s?\d{4}\s?\d{0,4}')
CARD_MASK_RE = re.compile(r'\d{6}\*{4,}\d{4}')
LONG_NUM_RE = re.compile(r'\b\d{8,}\b')
VS_SS_KS_RE = re.compile(r'\b[VvSsKk][Ss]\s*[:=]?\s*\d+', re.IGNORECASE)
DATE_INLINE_RE = re.compile(r'\b\d{1,2}[./]\d{1,2}[./]\d{2,4}\b')
LEGAL_SUFFIX_RE = re.compile(
    r'\b(s\.?\s?r\.?\s?o\.?|a\.?\s?s\.?|spol\.?|'
    r'ltd\.?|gmbh|inc\.?|corp\.?|llc|b\.?v\.?|n\.?v\.?|'
    r'se|ag|plc|oy|ab|sa|sas|srl|kft|sp\.?\s?z\.?\s?o\.?\s?o\.?)\s*$',
    re.IGNORECASE
)
STORE_NUM_PREFIX_RE = re.compile(r'^\d{3,6}[_\-\s]')
TRAILING_NUMS_RE = re.compile(r'\s+\d{2,}$')
DOMAIN_RE = re.compile(r'\b([a-zA-Z0-9-]+)\.(cz|sk|com|eu|net|org|io|de|at|hu|pl|co\.uk|com\.au)\b')


def clean_text(text: str) -> str:
    """Strip bank noise from raw transaction text."""
    if not text or not isinstance(text, str):
        return ''
    t = text.strip()
    t = IBAN_RE.sub('', t)
    t = CARD_MASK_RE.sub('', t)
    t = LONG_NUM_RE.sub('', t)
    t = VS_SS_KS_RE.sub('', t)
    t = DATE_INLINE_RE.sub('', t)
    # Collapse whitespace
    t = re.sub(r'\s+', ' ', t).strip()
    # Remove leading/trailing punctuation junk
    t = t.strip(' ,-/_~')
    return t


def normalize_merchant(text: str) -> str:
    """Normalize to canonical matching form."""
    if not text:
        return ''
    t = text.strip()
    # Remove store number prefix ("3520_SUPER ZOO" -> "SUPER ZOO")
    t = STORE_NUM_PREFIX_RE.sub('', t)
    # Remove legal suffixes
    t = LEGAL_SUFFIX_RE.sub('', t).strip()
    # Remove trailing numbers (branch IDs)
    t = TRAILING_NUMS_RE.sub('', t).strip()
    # Remove common prefixes from Revolut
    for prefix in ['Payment from ', 'To ', 'From ']:
        if t.startswith(prefix) and len(t) > len(prefix) + 2:
            t = t[len(prefix):]
    # Lowercase and collapse
    t = t.lower().strip()
    t = re.sub(r'\s+', ' ', t)
    # Strip remaining punctuation edges
    t = t.strip(' ,-/_~.*')
    return t


def extract_domain(text: str) -> str | None:
    """Extract domain root from text: 'alza.cz' -> 'alza'."""
    if not text:
        return None
    m = DOMAIN_RE.search(text)
    if m:
        return m.group(1).lower()
    return None


def get_merchant_text(row: pd.Series) -> str:
    """Pick best raw text for merchant extraction."""
    # Prefer partner field, fall back to description
    partner = str(row.get('partner', '') or '')
    desc = str(row.get('description', '') or '')
    if partner.strip():
        return partner.strip()
    if desc.strip():
        return desc.strip()
    return ''


# Apply to all transactions
txns['raw_merchant'] = txns.apply(get_merchant_text, axis=1)
txns['cleaned'] = txns['raw_merchant'].apply(clean_text)
txns['normalized'] = txns['cleaned'].apply(normalize_merchant)
txns['domain'] = txns['raw_merchant'].apply(extract_domain)

# Show before/after examples
sample = txns[txns.normalized != ''][['bank', 'raw_merchant', 'cleaned', 'normalized', 'domain']].drop_duplicates('raw_merchant').head(30)
print('=== Before/After Examples ===')
for _, row in sample.iterrows():
    parts = [f'  raw: {row.raw_merchant}', f'  clean: {row.cleaned}', f'  norm: {row.normalized}']
    if row.domain:
        parts.append(f'  domain: {row.domain}')
    print(f'[{row.bank}]')
    print('\n'.join(parts))
    print()

=== Before/After Examples ===
[SLSP]
  raw: Wolt
  clean: Wolt
  norm: wolt
  domain: nan

[SLSP]
  raw: Lejková Rebeka
  clean: Lejková Rebeka
  norm: lejková rebeka
  domain: nan

[SLSP]
  raw: NETFLIX INTERNATIONAL B.V
  clean: NETFLIX INTERNATIONAL B.V
  norm: netflix international
  domain: nan

[SLSP]
  raw: Lidl dakuje 165
  clean: Lidl dakuje 165
  norm: lidl dakuje
  domain: nan

[SLSP]
  raw: Vyšný Andrej
  clean: Vyšný Andrej
  norm: vyšný andrej
  domain: nan

[SLSP]
  raw: Voyo.sk 4691880802
  clean: Voyo.sk
  norm: voyo.sk
  domain: voyo

[SLSP]
  raw: ZSSK.SK
  clean: ZSSK.SK
  norm: zssk.sk
  domain: nan

[SLSP]
  raw: Dr.Max 156, RK Hrabovska
  clean: Dr.Max 156, RK Hrabovska
  norm: dr.max 156, rk hrabovska
  domain: nan

[SLSP]
  raw: ANEF, s.r.o.
  clean: ANEF, s.r.o.
  norm: anef
  domain: nan

[SLSP]
  raw: Bistro.sk, s. r. o.
  clean: Bistro.sk, s. r. o.
  norm: bistro.sk
  domain: bistro

[SLSP]
  raw: Lidl dakuje 226
  clean: Lidl dakuje 226
  norm: lidl dakuje

In [4]:
# Cell 4 — Merchant Extraction Analysis

non_empty = txns[txns.normalized != '']
empty = txns[txns.normalized == '']

print(f'Total transactions: {len(txns)}')
print(f'With merchant text: {len(non_empty)} ({len(non_empty)/len(txns)*100:.1f}%)')
print(f'Empty (bank fees, interest, etc.): {len(empty)} ({len(empty)/len(txns)*100:.1f}%)')
print()

# Frequency distribution
norm_counts = non_empty.normalized.value_counts()
print(f'Unique normalized merchant names: {len(norm_counts)}')
print(f'Names appearing 1x: {(norm_counts == 1).sum()}')
print(f'Names appearing 2-5x: {((norm_counts >= 2) & (norm_counts <= 5)).sum()}')
print(f'Names appearing 5+x: {(norm_counts > 5).sum()}')
print()

# Top 50 most common
print('=== Top 50 Merchants by Frequency ===')
for name, count in norm_counts.head(50).items():
    print(f'  {count:4d}x  {name}')
print()

# Show variant groups — same normalized form from different raw texts
print('=== Variant Groups (different raw → same normalized) ===')
variant_groups = non_empty.groupby('normalized')['raw_merchant'].apply(lambda x: sorted(set(x))).reset_index()
multi_variant = variant_groups[variant_groups.raw_merchant.apply(len) > 1].head(20)
for _, row in multi_variant.iterrows():
    print(f'  "{row.normalized}" ← {row.raw_merchant}')

Total transactions: 2741
With merchant text: 2670 (97.4%)
Empty (bank fees, interest, etc.): 71 (2.6%)

Unique normalized merchant names: 315
Names appearing 1x: 166
Names appearing 2-5x: 82
Names appearing 5+x: 67

=== Top 50 Merchants by Frequency ===
   824x  pocket eur roundups from eur
   115x  exchanged to eur
   111x  andrej vyšný
    60x  billa spol s ro
    55x  billa
    54x  lejková rebeka
    49x  apple.com/bill
    44x  super zoo
    43x  vyšný andrej
    43x  alza.cz
    43x  železničná spoločnosť slovensko - žssk
    43x  bolt
    39x  mcdonald's
    34x  terno
    33x  ba space
    32x  gubalová ivona,vyšný ľubomír
    29x  terno zahradni
    29x  sporenie na rezervu
    27x  wolt
    27x  zssk.sk
    26x  andrej vysny
    25x  bageterie boulevard
    25x  parkdots
    22x  lidl dakuje
    22x  pocket withdrawal
    18x  tesco ruzomberok
    16x  www.websupport.sk comfo
    15x  marek gavura
    14x  pocket eur savings from eur
    14x  exchanged to usd
    14x  cursor


In [5]:
# Cell 5 — Build Simulated Merchant List

# Simulate processing transactions chronologically, building merchant list from scratch
sorted_txns = txns.sort_values('date').reset_index(drop=True)

merchants: dict[str, int] = {}  # normalized_name -> merchant_id
aliases: dict[str, int] = {}    # alias -> merchant_id
next_id = 1
results = []  # (idx, merchant_id, method)

for idx, row in sorted_txns.iterrows():
    norm = row['normalized']
    if not norm:
        results.append((idx, None, 'empty'))
        continue
    
    # Exact match
    if norm in merchants:
        results.append((idx, merchants[norm], 'exact'))
        continue
    
    # New merchant
    mid = next_id
    next_id += 1
    merchants[norm] = mid
    aliases[norm] = mid
    results.append((idx, mid, 'new'))

results_df = pd.DataFrame(results, columns=['idx', 'merchant_id', 'method'])
method_counts = results_df.method.value_counts()

print('=== Exact Match Only (no fuzzy) ===')
print(f'Total transactions: {len(results_df)}')
for method, count in method_counts.items():
    print(f'  {method}: {count} ({count/len(results_df)*100:.1f}%)')
print(f'\nMerchants created: {len(merchants)}')
print(f'Match rate (exact only): {method_counts.get("exact", 0) / len(results_df) * 100:.1f}%')

# Show merchant list growth over time
cumulative_new = (results_df.method == 'new').cumsum()
print(f'\nMerchant list growth (every 200 txns):')
for i in range(0, len(cumulative_new), 200):
    print(f'  After {i:4d} txns: {cumulative_new.iloc[i]} unique merchants')
print(f'  Final ({len(cumulative_new)} txns): {cumulative_new.iloc[-1]} unique merchants')

=== Exact Match Only (no fuzzy) ===
Total transactions: 2741
  exact: 2355 (85.9%)
  new: 315 (11.5%)
  empty: 71 (2.6%)

Merchants created: 315
Match rate (exact only): 85.9%

Merchant list growth (every 200 txns):
  After    0 txns: 1 unique merchants
  After  200 txns: 71 unique merchants
  After  400 txns: 101 unique merchants
  After  600 txns: 136 unique merchants
  After  800 txns: 158 unique merchants
  After 1000 txns: 173 unique merchants
  After 1200 txns: 185 unique merchants
  After 1400 txns: 203 unique merchants
  After 1600 txns: 226 unique merchants
  After 1800 txns: 247 unique merchants
  After 2000 txns: 263 unique merchants
  After 2200 txns: 273 unique merchants
  After 2400 txns: 288 unique merchants
  After 2600 txns: 305 unique merchants
  Final (2741 txns): 315 unique merchants


In [6]:
# Cell 6 — Fuzzy Matching with RapidFuzz

# Test fuzzy matching on the unique normalized names
unique_names = sorted(merchants.keys())
print(f'Testing fuzzy matching across {len(unique_names)} unique normalized names\n')

# Find potential merges using different strategies and thresholds
def find_fuzzy_matches(names: list[str], scorer, threshold: int, label: str) -> list[tuple[str, str, int]]:
    matches = []
    seen = set()
    for name in names:
        if name in seen or len(name) < 3:
            continue
        results = process.extract(name, names, scorer=scorer, limit=5, score_cutoff=threshold)
        for match_name, score, _ in results:
            if match_name != name and match_name not in seen:
                matches.append((name, match_name, int(score)))
        seen.add(name)
    return matches

# Test token_set_ratio (handles word reordering)
for threshold in [95, 90, 85, 80]:
    matches = find_fuzzy_matches(unique_names, fuzz.token_set_ratio, threshold, 'token_set')
    print(f'token_set_ratio >= {threshold}: {len(matches)} potential merges')
    if matches:
        for a, b, score in matches[:10]:
            print(f'  {score}%  "{a}" ↔ "{b}"')
        if len(matches) > 10:
            print(f'  ... and {len(matches) - 10} more')
    print()

# Test partial_ratio (handles substrings: "Wolt Bratislava" ↔ "Wolt")
print('--- partial_ratio ---')
for threshold in [95, 90, 85]:
    matches = find_fuzzy_matches(unique_names, fuzz.partial_ratio, threshold, 'partial')
    print(f'partial_ratio >= {threshold}: {len(matches)} potential merges')
    if matches:
        for a, b, score in matches[:10]:
            print(f'  {score}%  "{a}" ↔ "{b}"')
        if len(matches) > 10:
            print(f'  ... and {len(matches) - 10} more')
    print()

Testing fuzzy matching across 315 unique normalized names

token_set_ratio >= 95: 42 potential merges
  100%  "13 verde" ↔ "verde"
  100%  "andrej slsp vysny" ↔ "andrej vysny"
  100%  "andrej vysny" ↔ "andrej vysny lisiak"
  100%  "andrej vyšný" ↔ "vyšný andrej"
  95%  "asian sun ba central" ↔ "asian sun, ba, central"
  100%  "bageterie boulevard" ↔ "bageterie boulevard c"
  100%  "billa" ↔ "billa spol s ro"
  100%  "billa" ↔ "billa spolsro"
  100%  "blaho" ↔ "kaviaren blaho"
  100%  "bratislava" ↔ "c007 bratislava"
  ... and 32 more

token_set_ratio >= 90: 49 potential merges
  100%  "13 verde" ↔ "verde"
  100%  "andrej slsp vysny" ↔ "andrej vysny"
  100%  "andrej vysny" ↔ "andrej vysny lisiak"
  100%  "andrej vyšný" ↔ "vyšný andrej"
  95%  "asian sun ba central" ↔ "asian sun, ba, central"
  100%  "bageterie boulevard" ↔ "bageterie boulevard c"
  100%  "billa" ↔ "billa spol s ro"
  100%  "billa" ↔ "billa spolsro"
  100%  "blaho" ↔ "kaviaren blaho"
  100%  "bratislava" ↔ "c007 bratisla

In [ ]:
# Cell 7 — Domain Extraction

domain_txns = txns[txns.domain.notna()].copy()
print(f'Transactions with domain patterns: {len(domain_txns)} / {len(txns)} ({len(domain_txns)/len(txns)*100:.1f}%)\n')

# Show all domain extractions
domain_groups = domain_txns.groupby('domain').agg(
    count=('domain', 'size'),
    raw_examples=('raw_merchant', lambda x: sorted(set(x))[:3]),
    normalized=('normalized', 'first'),
).sort_values('count', ascending=False)

print('=== Extracted Domains ===')
for domain, row in domain_groups.iterrows():
    print(f'  {domain} ({row["count"]}x): raw={row.raw_examples}, norm="{row.normalized}"')

# Check if domain roots match any existing normalized names
print('\n=== Domain → Merchant Name Matching ===')
all_norms = set(merchants.keys())
for domain in domain_groups.index:
    # Check if domain appears as substring in any normalized name
    matches = [n for n in all_norms if domain in n and n != domain]
    if matches:
        print(f'  domain "{domain}" matches normalized names: {matches}')

In [ ]:
# Cell 8 — Full Pipeline Simulation

def resolve_merchant(
    raw_text: str,
    known_merchants: dict[str, int],
    alias_map: dict[str, int],
    next_id_ref: list[int],
    fuzzy_threshold_token: int = 90,
    fuzzy_threshold_partial: int = 90,
) -> tuple[int | None, str | None, str]:
    """
    Resolve a merchant from raw transaction text.
    Returns (merchant_id, new_merchant_name_if_created, method).
    """
    cleaned = clean_text(raw_text)
    norm = normalize_merchant(cleaned)
    
    if not norm:
        return (None, None, 'empty')
    
    # 1. Exact alias lookup
    if norm in alias_map:
        return (alias_map[norm], None, 'exact_alias')
    
    # 2. Domain extraction + lookup
    domain = extract_domain(raw_text)
    if domain and domain in alias_map:
        # Register this normalized form as alias
        mid = alias_map[domain]
        alias_map[norm] = mid
        return (mid, None, 'domain')
    # Also check if domain matches any known merchant name
    if domain:
        for known_name, mid in known_merchants.items():
            if domain in known_name or known_name in domain:
                alias_map[norm] = mid
                if domain not in alias_map:
                    alias_map[domain] = mid
                return (mid, None, 'domain_substring')
    
    # 3. Fuzzy matching — token_set_ratio (word reordering)
    known_names = list(known_merchants.keys())
    if known_names and len(norm) >= 3:
        best = process.extractOne(norm, known_names, scorer=fuzz.token_set_ratio, score_cutoff=fuzzy_threshold_token)
        if best:
            match_name, score, _ = best
            mid = known_merchants[match_name]
            alias_map[norm] = mid
            return (mid, None, f'fuzzy_token({score:.0f})')
    
    # 4. Fuzzy matching — partial_ratio (substring matching)
    # Only for longer names to avoid false positives
    if known_names and len(norm) >= 5:
        best = process.extractOne(norm, known_names, scorer=fuzz.partial_ratio, score_cutoff=fuzzy_threshold_partial)
        if best:
            match_name, score, _ = best
            # Extra guard: matched name must be at least 60% the length of query
            if len(match_name) >= len(norm) * 0.6:
                mid = known_merchants[match_name]
                alias_map[norm] = mid
                return (mid, None, f'fuzzy_partial({score:.0f})')
    
    # 5. No match → create new merchant
    mid = next_id_ref[0]
    next_id_ref[0] += 1
    known_merchants[norm] = mid
    alias_map[norm] = mid
    # Also register domain as alias if present
    if domain and domain not in alias_map:
        alias_map[domain] = mid
    return (mid, norm, 'new')


# Run full pipeline on all transactions (chronological order)
sorted_txns = txns.sort_values('date').reset_index(drop=True)

pipeline_merchants: dict[str, int] = {}
pipeline_aliases: dict[str, int] = {}
pipeline_next_id = [1]
pipeline_results = []

for idx, row in sorted_txns.iterrows():
    raw = get_merchant_text(row)
    mid, new_name, method = resolve_merchant(
        raw, pipeline_merchants, pipeline_aliases, pipeline_next_id
    )
    pipeline_results.append({
        'idx': idx,
        'raw': raw,
        'merchant_id': mid,
        'new_name': new_name,
        'method': method,
        'bank': row['bank'],
    })

pr_df = pd.DataFrame(pipeline_results)

# Classify methods into categories
def method_category(m: str) -> str:
    if m == 'exact_alias': return 'Exact alias'
    elif m.startswith('domain'): return 'Domain extraction'
    elif m.startswith('fuzzy_token'): return 'Fuzzy (token_set)'
    elif m.startswith('fuzzy_partial'): return 'Fuzzy (partial)'
    elif m == 'new': return 'New merchant'
    elif m == 'empty': return 'Unresolvable'
    return m

pr_df['method_cat'] = pr_df.method.apply(method_category)

print('=== Full Pipeline Results ===')
print(f'Total transactions: {len(pr_df)}')
print(f'Merchants created: {len(pipeline_merchants)}')
print(f'Aliases registered: {len(pipeline_aliases)}')
print()

cat_counts = pr_df.method_cat.value_counts()
print('Resolution breakdown:')
for cat, count in cat_counts.items():
    print(f'  {cat:25s} {count:5d}  ({count/len(pr_df)*100:.1f}%)')

resolved = pr_df[pr_df.method != 'empty']
matched = resolved[resolved.method != 'new']
print(f'\nOverall match rate: {len(matched)}/{len(resolved)} = {len(matched)/len(resolved)*100:.1f}% of non-empty transactions')
print(f'Total resolved: {len(resolved)}/{len(pr_df)} = {len(resolved)/len(pr_df)*100:.1f}% of all transactions')

# Per-bank breakdown
print('\n--- Per Bank ---')
for bank in ['SLSP', 'Revolut']:
    bank_df = pr_df[pr_df.bank == bank]
    bank_resolved = bank_df[bank_df.method != 'empty']
    bank_matched = bank_resolved[bank_resolved.method != 'new']
    print(f'{bank}: {len(bank_matched)}/{len(bank_resolved)} matched ({len(bank_matched)/max(len(bank_resolved),1)*100:.1f}%), '
          f'{len(bank_df) - len(bank_resolved)} empty')

In [ ]:
# Cell 9 — Quality Assessment

import random
random.seed(42)

# Build reverse lookup: merchant_id -> canonical name
id_to_name: dict[int, str] = {v: k for k, v in pipeline_merchants.items()}

# 1. Sample 20 random resolved (non-new, non-empty) assignments
matched_rows = pr_df[(pr_df.method != 'empty') & (pr_df.method != 'new')].copy()
if len(matched_rows) >= 20:
    sample_idx = random.sample(range(len(matched_rows)), 20)
    sample_matched = matched_rows.iloc[sample_idx]
else:
    sample_matched = matched_rows

print('=== Sample 20 Matched Assignments (manual review) ===')
for _, row in sample_matched.iterrows():
    merchant_name = id_to_name.get(row['merchant_id'], '?')
    print(f'  [{row.method:25s}] "{row.raw}" → merchant "{merchant_name}" (id={row.merchant_id})')

# 2. List all fuzzy matches for review
fuzzy_rows = pr_df[pr_df.method.str.startswith('fuzzy')].copy()
print(f'\n=== All Fuzzy Matches ({len(fuzzy_rows)} total) ===')
fuzzy_unique = fuzzy_rows.drop_duplicates(subset=['raw', 'merchant_id'])
for _, row in fuzzy_unique.iterrows():
    merchant_name = id_to_name.get(row['merchant_id'], '?')
    print(f'  [{row.method:25s}] "{row.raw}" → "{merchant_name}"')

# 3. Merchants that were created but possibly should have been merged
print(f'\n=== Possible Missed Merges ===')
merchant_names = list(pipeline_merchants.keys())
missed_merges = []
checked = set()
for name in merchant_names:
    if name in checked or len(name) < 3:
        continue
    results = process.extract(name, merchant_names, scorer=fuzz.token_set_ratio, limit=3, score_cutoff=75)
    for match, score, _ in results:
        if match != name and (match, name) not in checked:
            missed_merges.append((name, match, int(score)))
            checked.add((name, match))
            checked.add((match, name))
    checked.add(name)

missed_merges.sort(key=lambda x: -x[2])
for a, b, score in missed_merges[:30]:
    print(f'  {score}%  "{a}" ↔ "{b}"')
if len(missed_merges) > 30:
    print(f'  ... and {len(missed_merges) - 30} more')

# 4. Unresolved transactions review
empty_rows = pr_df[pr_df.method == 'empty']
print(f'\n=== Unresolvable Transactions ({len(empty_rows)}) ===')
# Show the original partner/description for empty rows
for _, row in empty_rows.head(20).iterrows():
    orig = sorted_txns.iloc[row['idx']]
    print(f'  [{orig.bank}] partner="{orig.partner}", desc="{orig.description}", type="{orig.tx_type}"')

In [ ]:
# Cell 10 — Conclusions & Next Steps

total = len(pr_df)
empty_count = len(pr_df[pr_df.method == 'empty'])
new_count = len(pr_df[pr_df.method == 'new'])
exact_count = len(pr_df[pr_df.method == 'exact_alias'])
domain_count = len(pr_df[pr_df.method_cat == 'Domain extraction'])
fuzzy_token_count = len(pr_df[pr_df.method_cat == 'Fuzzy (token_set)'])
fuzzy_partial_count = len(pr_df[pr_df.method_cat == 'Fuzzy (partial)'])
resolved_count = total - empty_count
matched_count = resolved_count - new_count

print('=' * 60)
print('MERCHANT EXTRACTION & MATCHING — RESULTS SUMMARY')
print('=' * 60)
print(f'''
Total transactions:       {total}
With merchant text:       {resolved_count} ({resolved_count/total*100:.1f}%)
Unresolvable (empty):     {empty_count} ({empty_count/total*100:.1f}%)

--- Resolution Methods ---
Exact alias match:        {exact_count} ({exact_count/total*100:.1f}%)
Domain extraction:        {domain_count} ({domain_count/total*100:.1f}%)
Fuzzy (token_set >= 90):  {fuzzy_token_count} ({fuzzy_token_count/total*100:.1f}%)
Fuzzy (partial >= 90):    {fuzzy_partial_count} ({fuzzy_partial_count/total*100:.1f}%)
New merchant created:     {new_count} ({new_count/total*100:.1f}%)

--- Key Metrics ---
Match rate (of non-empty): {matched_count}/{resolved_count} = {matched_count/max(resolved_count,1)*100:.1f}%
Unique merchants created:  {len(pipeline_merchants)}
Aliases registered:        {len(pipeline_aliases)}
''')

print('--- Observations ---')
print('''
1. Normalization alone (lowercase, strip suffixes/store numbers) handles the bulk of matching.
2. Exact alias matching is by far the most effective method — most transactions
   are repeat visits to the same merchants.
3. Domain extraction is a small but precise signal.
4. Fuzzy matching catches variant spellings but needs careful thresholds
   to avoid false positives. token_set_ratio >= 90 is safer than partial_ratio.
5. "Empty" transactions are bank fees, interest, savings transfers — expected.

--- Recommended Production Thresholds ---
- Auto-match: exact alias OR fuzzy >= 95 → auto-assign
- Suggest: fuzzy 85-94 → suggest to user for confirmation  
- New: no match → auto-create new merchant
- Learn: manual user assignment → create alias for future auto-match

--- Next Steps for Production ---
1. merchant_aliases table: (user_id, alias, merchant_id) with unique constraint
2. Run inline during import (not separate ML service)
3. UI: merchant dropdown with "create new" option on transaction edit
4. Manual assignment creates alias → improves future matching
5. No embeddings needed for this volume — rapidfuzz is sufficient
6. Consider embeddings only if: multilingual matching needed OR >10k unique merchants
''')

---

# Counterparty Classification Extension

Cells A-E add entity type classification (Merchant vs Person), improved normalization, and cross-source matching.

In [ ]:
# Cell A — Entity Type Scorer
#
# Weighted signal scoring to classify transactions as person vs merchant.
# Positive score → person, negative → merchant.

import unicodedata

def ascii_fold(text: str) -> str:
    """Remove diacritics: 'Vyšný' → 'Vysny'."""
    if not text:
        return ''
    nfkd = unicodedata.normalize('NFKD', text)
    return ''.join(c for c in nfkd if not unicodedata.combining(c))

# --- Scoring patterns ---
LEGAL_SUFFIX_SCORE_RE = re.compile(
    r'\b(s\.?\s?r\.?\s?o\.?|a\.?\s?s\.?|spol\.?\s*(s\.?\s?r\.?\s?o\.?)?|'
    r'ltd\.?|gmbh|inc\.?|corp\.?|llc|b\.?\s?v\.?|n\.?\s?v\.?|'
    r'se|ag|plc|oy|ab|sa|sas|srl|kft|sp\.?\s?z\.?\s?o\.?\s?o\.?)\b',
    re.IGNORECASE
)
DOMAIN_SCORE_RE = re.compile(
    r'[a-zA-Z0-9-]+\.(cz|sk|com|eu|net|org|io|de|at|hu|pl|co\.uk)'
)
STORE_NUM_SCORE_RE = re.compile(r'(^\d{3,6}[_\-\s]|\b\d{3,4}\b)')
REVOLUT_PERSON_PREFIX_RE = re.compile(
    r'^(Payment from|Transfer from|Transfer to|To|From)\s+', re.IGNORECASE
)

# Proper-case 2-word name (Slovak diacritics included)
PROPER_CASE_2WORD_RE = re.compile(
    r'^[A-ZÁÄČĎÉÍĹĽŇÓÔŔŠŤÚÝŽ][a-záäčďéíĺľňóôŕšťúýž]+\s+'
    r'[A-ZÁÄČĎÉÍĹĽŇÓÔŔŠŤÚÝŽ][a-záäčďéíĺľňóôŕšťúýž]+$'
)
ALL_CAPS_2WORD_RE = re.compile(r'^[A-Z]{2,}\s+[A-Z]{2,}$')
# ALL CAPS abbreviation (both words ≤5 chars) — likely org/brand, not person
ABBREV_2WORD_RE = re.compile(r'^[A-Z]{2,5}\s+[A-Z]{2,5}$')

# Internal/system transaction patterns — classify as merchant
INTERNAL_TX_RE = re.compile(
    r'^(To pocket|Pocket Withdrawal|Exchanged to|Closing transaction|'
    r'Sporenie na rezervu|Sporenie$|SPACE účet|Referral reward)',
    re.IGNORECASE
)

# TX type sets
PERSON_TX_TYPES = {
    'prijatá sepa platba', 'sepa platba (elektronicky)',
    'sepa platba trvalým príkazom',
    'bezhotovostný vklad', 'bezhotovostný vklad–okamžitá platba',
    'platobný príkaz na úhradu / fit 2.0 (eb)',
    'okamžitá platba (eb)', 'okamžitá platba trvalým príkazom',
    'okamžitá platba (elektronicky)',
    'trvalý platobný príkaz na úhradu',
    'topup',
}
MERCHANT_TX_TYPES = {
    'platba kartou', 'card payment',
    'inkaso (platiteľ)', 'exchange', 'card refund', 'reward',
}


def classify_entity(row: pd.Series) -> tuple[str, float, float]:
    """
    Classify transaction as 'person', 'merchant', or 'unknown'.
    Returns (entity_type, score, confidence).
    """
    score = 0.0
    partner = str(row.get('partner', '') or '').strip()
    raw = str(row.get('raw_merchant', '') or '').strip()
    tx_type = str(row.get('tx_type', '') or '').strip().lower()
    partner_iban = str(row.get('partner_iban', '') or '').strip()

    if not partner and not raw:
        return ('empty', 0.0, 1.0)

    # Hard override: legal suffix is definitive → always merchant
    if LEGAL_SUFFIX_SCORE_RE.search(partner):
        return ('merchant', -3.0, 0.95)

    # Early exit: internal/system transactions → merchant
    if INTERNAL_TX_RE.match(partner):
        return ('merchant', -1.0, 0.85)

    # Hard rule: single-word partner (no spaces) → merchant
    has_person_prefix = bool(REVOLUT_PERSON_PREFIX_RE.match(partner))
    if ' ' not in partner.strip() and partner.strip() and not has_person_prefix:
        return ('merchant', -0.5, 0.80)

    # Hard rule: ALL CAPS abbreviation 2-word (both ≤5 chars) → merchant
    # Catches orgs like "BA SPACE", "FEI STU" that aren't person names
    text_to_check = partner if partner else raw
    if ABBREV_2WORD_RE.match(text_to_check):
        return ('merchant', -1.5, 0.85)

    # --- Person signals ---

    # Partner IBAN present (+3.0)
    if partner_iban and len(partner_iban) > 10:
        score += 3.0

    # TX type is SEPA/transfer/topup (+2.5)
    if tx_type in PERSON_TX_TYPES:
        score += 2.5

    # Revolut "Payment from" / "Transfer from/to" prefix (+2.0)
    if has_person_prefix:
        score += 2.0

    # 2-word proper-case name (+1.0) — only if transfer-type tx (not card payment)
    if tx_type not in MERCHANT_TX_TYPES:
        if PROPER_CASE_2WORD_RE.match(text_to_check):
            score += 1.0
        # ALL CAPS 2-word (long words → likely person name like "ANDREJ VYSNY")
        if ALL_CAPS_2WORD_RE.match(text_to_check):
            score += 0.8

    # --- Merchant signals ---

    # Domain pattern (-1.5)
    if DOMAIN_SCORE_RE.search(partner):
        score -= 1.5

    # Store/branch number (-1.5)
    if STORE_NUM_SCORE_RE.search(partner):
        score -= 1.5

    # Card payment tx type (-1.0)
    if tx_type in MERCHANT_TX_TYPES:
        score -= 1.0

    # --- Thresholds ---
    if score > 2.0:
        conf = min(0.85 + (score - 2.0) * 0.03, 0.99)
        return ('person', score, round(conf, 2))
    elif score > 0.5:
        return ('person', score, 0.70)
    elif score < -2.0:
        conf = min(0.85 + (abs(score) - 2.0) * 0.03, 0.99)
        return ('merchant', score, round(conf, 2))
    elif score < -0.5:
        return ('merchant', score, 0.70)
    else:
        return ('unknown', score, 0.50)


# --- Run on all transactions ---
classifications = txns.apply(classify_entity, axis=1, result_type='expand')
classifications.columns = ['entity_type', 'entity_score', 'entity_confidence']
txns = pd.concat([txns, classifications], axis=1)

# --- Reports ---
print('=== Entity Type Distribution ===')
type_counts = txns.entity_type.value_counts()
for t, count in type_counts.items():
    print(f'  {t:10s}: {count:5d} ({count/len(txns)*100:.1f}%)')

unknown_pct = type_counts.get('unknown', 0) / len(txns) * 100
empty_pct = type_counts.get('empty', 0) / len(txns) * 100
classified_pct = 100.0 - unknown_pct - empty_pct
print(f'\nClassified (non-unknown, non-empty): {classified_pct:.1f}%')

print('\n=== Distribution by Bank ===')
for bank in ['SLSP', 'Revolut']:
    bank_df = txns[txns.bank == bank]
    print(f'\n  {bank} ({len(bank_df)} txns):')
    for t, count in bank_df.entity_type.value_counts().items():
        print(f'    {t:10s}: {count:5d} ({count/len(bank_df)*100:.1f}%)')

print('\n=== Confidence Distribution ===')
for t in ['person', 'merchant', 'unknown']:
    subset = txns[txns.entity_type == t]
    if len(subset) > 0:
        print(f'\n  {t} (n={len(subset)}):')
        bins = [0.4, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0]
        for i in range(len(bins) - 1):
            count = ((subset.entity_confidence >= bins[i]) & (subset.entity_confidence < bins[i+1])).sum()
            if count > 0:
                bar = '#' * min(count // 2, 40)
                print(f'    [{bins[i]:.2f}-{bins[i+1]:.2f}): {count:4d} {bar}')

print('\n=== All UNKNOWN Cases ===')
unknown = txns[txns.entity_type == 'unknown'][
    ['bank', 'partner', 'tx_type', 'entity_score', 'raw_merchant']
].drop_duplicates('raw_merchant')
print(f'Total unique UNKNOWN: {len(unknown)}')
for _, row in unknown.iterrows():
    print(f'  [{row.bank}] score={row.entity_score:+.1f} '
          f'partner="{row.partner}" type="{row.tx_type}"')

# Person detection review
print('\n=== All Detected Persons — Manual Review ===')
persons = txns[txns.entity_type == 'person'][
    ['bank', 'partner', 'entity_score', 'entity_confidence']
].drop_duplicates('partner').sort_values('entity_score', ascending=False)
print(f'Total unique person names: {len(persons)}')
for _, row in persons.iterrows():
    print(f'  [{row.bank:7s}] score={row.entity_score:+.1f} conf={row.entity_confidence:.2f} "{row.partner}"')

In [ ]:
# Cell B — Person Name Normalization
#
# Canonical form: first-seen form preserved (with diacritics when available).
# Generates reversed-order + ASCII-folded aliases for matching.

def normalize_person(text: str) -> tuple[str, list[str]]:
    """
    Normalize a person name and generate matching aliases.
    Returns (canonical_name, [aliases]).
    """
    if not text or not isinstance(text, str):
        return ('', [])

    t = text.strip()

    # Strip Revolut prefixes
    for prefix in ['Payment from ', 'To ', 'From ']:
        if t.startswith(prefix) and len(t) > len(prefix) + 2:
            t = t[len(prefix):]

    # Title case (handle ALL CAPS input)
    if t.isupper():
        t = t.title()

    canonical = t.strip()

    # Generate aliases
    aliases = set()
    lower = canonical.lower()
    aliases.add(lower)

    # ASCII-folded lowercase
    ascii_lower = ascii_fold(lower)
    aliases.add(ascii_lower)

    # Reversed word order (handles "Surname First" ↔ "First Surname")
    words = lower.split()
    if len(words) >= 2:
        reversed_name = ' '.join(reversed(words))
        aliases.add(reversed_name)
        aliases.add(ascii_fold(reversed_name))

    return (canonical, sorted(aliases))


# --- Apply to all detected persons ---
persons = txns[txns.entity_type == 'person'].copy()
person_results = persons['raw_merchant'].apply(normalize_person)
persons['person_canonical'] = person_results.apply(lambda x: x[0])
persons['person_aliases'] = person_results.apply(lambda x: x[1])

print(f'=== Person Normalization ({len(persons)} transactions) ===')
print(f'Unique raw person names: {persons.raw_merchant.nunique()}')
print(f'Unique canonical names:  {persons.person_canonical.nunique()}')

print('\n=== Before/After for All Detected Persons ===')
unique_persons = persons.drop_duplicates('raw_merchant')[
    ['bank', 'raw_merchant', 'person_canonical', 'person_aliases']
]
for _, row in unique_persons.iterrows():
    print(f'  [{row.bank:7s}] "{row.raw_merchant}"')
    print(f'           -> canonical: "{row.person_canonical}"')
    print(f'           -> aliases:   {row.person_aliases}')

In [ ]:
# Cell C — Cross-Source Person Matching
#
# Test the critical scenario: same person appearing from different banks.
# "Vyšný Andrej" (SLSP) ↔ "ANDREJ VYSNY" (Revolut) should match via aliases.

def build_alias_pool(person_df: pd.DataFrame) -> dict[str, str]:
    """Build alias -> canonical mapping from person DataFrame."""
    pool: dict[str, str] = {}
    for _, row in person_df.drop_duplicates('person_canonical').iterrows():
        canonical = row['person_canonical']
        for alias in row['person_aliases']:
            pool[alias] = canonical
    return pool

slsp_persons_df = persons[persons.bank == 'SLSP']
revolut_persons_df = persons[persons.bank == 'Revolut']

slsp_unique = set(slsp_persons_df['person_canonical'].unique())
revolut_unique = set(revolut_persons_df['person_canonical'].unique())

print(f'=== Cross-Source Person Matching ===')
print(f'SLSP unique persons:    {len(slsp_unique)}')
print(f'Revolut unique persons: {len(revolut_unique)}')

slsp_pool = build_alias_pool(slsp_persons_df)
revolut_pool = build_alias_pool(revolut_persons_df)

# Find cross-source matches via alias overlap
cross_matches = []
for alias, slsp_canonical in slsp_pool.items():
    if alias in revolut_pool:
        rev_canonical = revolut_pool[alias]
        cross_matches.append((slsp_canonical, rev_canonical, alias))

# Deduplicate by (slsp, revolut) pair
seen_pairs: set[tuple[str, str]] = set()
unique_matches = []
for slsp_c, rev_c, alias in cross_matches:
    pair = (slsp_c, rev_c)
    if pair not in seen_pairs:
        seen_pairs.add(pair)
        unique_matches.append((slsp_c, rev_c, alias))

print(f'\nAlias-based cross-source matches: {len(unique_matches)}')
for slsp_c, rev_c, alias in unique_matches:
    print(f'  SLSP "{slsp_c}" <-> Revolut "{rev_c}" (via alias "{alias}")')

# Fuzzy matching for remaining unmatched persons
matched_slsp = {m[0] for m in unique_matches}
matched_rev = {m[1] for m in unique_matches}
slsp_unmatched = [c for c in slsp_unique if c not in matched_slsp]
revolut_unmatched = [c for c in revolut_unique if c not in matched_rev]

if slsp_unmatched and revolut_unmatched:
    print(f'\n=== Fuzzy Cross-Source (token_set_ratio >= 80) ===')
    print(f'Unmatched: {len(slsp_unmatched)} SLSP, {len(revolut_unmatched)} Revolut')
    slsp_ascii = {ascii_fold(n).lower(): n for n in slsp_unmatched}
    rev_ascii = {ascii_fold(n).lower(): n for n in revolut_unmatched}

    fuzzy_cross = []
    for s_ascii, s_orig in slsp_ascii.items():
        best = process.extractOne(
            s_ascii, list(rev_ascii.keys()),
            scorer=fuzz.token_set_ratio, score_cutoff=80
        )
        if best:
            r_ascii_match, score, _ = best
            r_orig = rev_ascii[r_ascii_match]
            fuzzy_cross.append((s_orig, r_orig, int(score)))
            print(f'  {score}% SLSP "{s_orig}" <-> Revolut "{r_orig}"')

    if not fuzzy_cross:
        print('  (no additional fuzzy matches)')

# --- Full person detection review ---
print(f'\n=== All Detected Persons — Manual Review ===')
all_person_names = persons.drop_duplicates('raw_merchant')[
    ['bank', 'raw_merchant', 'entity_score', 'entity_confidence']
].sort_values('entity_score', ascending=False)
print(f'Total unique person names: {len(all_person_names)}')
for _, row in all_person_names.iterrows():
    print(f'  [{row.bank:7s}] score={row.entity_score:+.1f} '
          f'conf={row.entity_confidence:.2f} "{row.raw_merchant}"')

In [ ]:
# Cell D — Extended Merchant Normalization
#
# Improvements over baseline: location stripping, payment processor prefixes,
# URL-to-brand extraction. Compare unique count reduction.

LOCATION_AFTER_COMMA_RE = re.compile(r',\s*[A-Z]{2}[\s,].*$|,\s*\w+$', re.IGNORECASE)
PAYMENT_PROCESSOR_RE = re.compile(
    r'^(GOPAY\s*\*|Payout\s*\*|24-pay\s*\*|SumUp\s*\*)\s*', re.IGNORECASE
)
URL_BRAND_RE = re.compile(
    r'^(www\.)?([a-zA-Z0-9-]+)\.(cz|sk|com|eu|net|org|io|de|at|hu|pl|co\.uk)(/.*)?$',
    re.IGNORECASE
)
DM_STORE_RE = re.compile(r'^(dm)\s+\d+\s+', re.IGNORECASE)
CXXX_PREFIX_RE = re.compile(r'^C\d{3}\s+')
SPOL_S_RO_RE = re.compile(r'\bSPOL\s+S\s+RO\b', re.IGNORECASE)


def normalize_merchant_v2(text: str) -> str:
    """Enhanced merchant normalization with location/processor/URL handling."""
    if not text:
        return ''
    t = text.strip()

    # Store number prefix
    t = STORE_NUM_PREFIX_RE.sub('', t)

    # C-prefix codes ("C058 Bratislava")
    t = CXXX_PREFIX_RE.sub('', t).strip()

    # Payment processor prefix
    t = PAYMENT_PROCESSOR_RE.sub('', t).strip()

    # URL-to-brand ("APPLE.COM/BILL" -> "Apple")
    url_match = URL_BRAND_RE.match(t)
    if url_match:
        t = url_match.group(2)

    # Legal suffixes
    t = LEGAL_SUFFIX_RE.sub('', t).strip()

    # "SPOL S RO" variant
    t = SPOL_S_RO_RE.sub('', t).strip()

    # Trailing branch numbers
    t = TRAILING_NUMS_RE.sub('', t).strip()

    # Location after comma: "KAUFLAND 2220, BA, TRNA" -> "KAUFLAND"
    t = LOCATION_AFTER_COMMA_RE.sub('', t).strip()

    # DM store: "dm 315 BA Central" -> "Dm"
    dm_match = DM_STORE_RE.match(t)
    if dm_match:
        t = dm_match.group(1)

    # Strip Revolut person prefixes (for merchants that slip through)
    for prefix in ['Payment from ', 'To ', 'From ']:
        if t.startswith(prefix) and len(t) > len(prefix) + 2:
            t = t[len(prefix):]

    # Title case for ALL CAPS
    if t.isupper() and len(t) > 1:
        t = t.title()

    # Lowercase matching key
    t_lower = t.lower().strip()
    t_lower = re.sub(r'\s+', ' ', t_lower)
    t_lower = t_lower.strip(' ,-/_~.*')
    return t_lower


def generate_merchant_aliases(raw: str, normalized: str) -> list[str]:
    """Generate alias variants for a merchant."""
    aliases = set()
    aliases.add(normalized)

    raw_lower = raw.lower().strip()
    if raw_lower:
        aliases.add(raw_lower)

    domain = extract_domain(raw)
    if domain:
        aliases.add(domain)

    # Brand root (first word if multi-word and long enough)
    words = normalized.split()
    if len(words) >= 2 and len(words[0]) >= 3:
        aliases.add(words[0])

    return sorted(aliases)


# --- Apply to merchant transactions ---
merchants_df = txns[txns.entity_type == 'merchant'].copy()
merchants_df['norm_v2'] = merchants_df['cleaned'].apply(normalize_merchant_v2)

baseline_unique = merchants_df['normalized'].nunique()
improved_unique = merchants_df['norm_v2'].nunique()

print('=== Extended Merchant Normalization ===')
print(f'Merchant transactions: {len(merchants_df)}')
print(f'Baseline unique names: {baseline_unique}')
print(f'Improved unique names: {improved_unique}')
print(f'Reduction: {baseline_unique - improved_unique} fewer '
      f'({(baseline_unique - improved_unique)/max(baseline_unique,1)*100:.1f}%)')

# Show new merges
print('\n=== New Merges from Improved Normalization ===')
merged_groups = merchants_df.groupby('norm_v2').agg(
    baseline_norms=('normalized', lambda x: sorted(set(x))),
    raw_variants=('raw_merchant', lambda x: sorted(set(x))),
    count=('raw_merchant', 'size'),
).reset_index()

new_merges = merged_groups[merged_groups.baseline_norms.apply(len) > 1]
for _, row in new_merges.iterrows():
    print(f'  "{row.norm_v2}" merges baseline names: {row.baseline_norms}')
    print(f'    raw variants: {row.raw_variants[:5]}')

# Show changed normalizations
print('\n=== Changed Normalizations (sample) ===')
changed = merchants_df[merchants_df.normalized != merchants_df.norm_v2][
    ['bank', 'raw_merchant', 'normalized', 'norm_v2']
].drop_duplicates('raw_merchant')
print(f'Total changed: {len(changed)}')
for _, row in changed.head(30).iterrows():
    print(f'  [{row.bank}] "{row.raw_merchant}"')
    print(f'    baseline: "{row.normalized}" -> improved: "{row.norm_v2}"')

In [ ]:
# Cell E — Full Pipeline with Person/Merchant Split
#
# Classify -> route to type-specific normalization -> match against
# type-specific counterparty pools. Comprehensive reporting.

import random
random.seed(42)

# --- Counterparty pools ---
person_pool: dict[str, int] = {}       # canonical -> id
person_alias_map: dict[str, int] = {}  # alias -> id
merchant_pool_v2: dict[str, int] = {}  # normalized -> id
merchant_alias_v2: dict[str, int] = {} # alias -> id
cp_info: dict[int, dict] = {}          # id -> {name, type}
next_cp_id = [1]


def resolve_person_cp(
    raw: str, canonical: str, aliases: list[str]
) -> tuple[int, str]:
    """Match or create a person counterparty. Returns (id, method)."""
    # Exact alias lookup
    for alias in aliases:
        if alias in person_alias_map:
            return (person_alias_map[alias], 'exact_alias')

    # Fuzzy on ASCII-folded canonical
    ascii_canonical = ascii_fold(canonical).lower()
    if person_pool:
        known = list(person_pool.keys())
        known_ascii = [ascii_fold(k).lower() for k in known]
        best = process.extractOne(
            ascii_canonical, known_ascii,
            scorer=fuzz.token_set_ratio, score_cutoff=85
        )
        if best:
            _, score, idx = best
            match_canonical = known[idx]
            pid = person_pool[match_canonical]
            for alias in aliases:
                person_alias_map[alias] = pid
            return (pid, f'fuzzy_person({score:.0f})')

    # New person
    pid = next_cp_id[0]
    next_cp_id[0] += 1
    person_pool[canonical] = pid
    for alias in aliases:
        person_alias_map[alias] = pid
    cp_info[pid] = {'name': canonical, 'type': 'person'}
    return (pid, 'new')


def resolve_merchant_cp(raw: str, cleaned: str) -> tuple[int | None, str]:
    """Match or create a merchant counterparty. Returns (id, method)."""
    norm = normalize_merchant_v2(cleaned)
    if not norm:
        return (None, 'empty')

    # Exact alias
    if norm in merchant_alias_v2:
        return (merchant_alias_v2[norm], 'exact_alias')

    # Domain lookup
    domain = extract_domain(raw)
    if domain and domain in merchant_alias_v2:
        mid = merchant_alias_v2[domain]
        merchant_alias_v2[norm] = mid
        return (mid, 'domain')
    if domain:
        for known_name, mid in merchant_pool_v2.items():
            if domain in known_name or known_name == domain:
                merchant_alias_v2[norm] = mid
                if domain not in merchant_alias_v2:
                    merchant_alias_v2[domain] = mid
                return (mid, 'domain_substring')

    # Fuzzy token_set_ratio
    known_names = list(merchant_pool_v2.keys())
    if known_names and len(norm) >= 3:
        best = process.extractOne(
            norm, known_names,
            scorer=fuzz.token_set_ratio, score_cutoff=90
        )
        if best:
            match_name, score, _ = best
            mid = merchant_pool_v2[match_name]
            merchant_alias_v2[norm] = mid
            return (mid, f'fuzzy_token({score:.0f})')

    # Fuzzy partial_ratio
    if known_names and len(norm) >= 5:
        best = process.extractOne(
            norm, known_names,
            scorer=fuzz.partial_ratio, score_cutoff=90
        )
        if best:
            match_name, score, _ = best
            if len(match_name) >= len(norm) * 0.6:
                mid = merchant_pool_v2[match_name]
                merchant_alias_v2[norm] = mid
                return (mid, f'fuzzy_partial({score:.0f})')

    # New merchant
    mid = next_cp_id[0]
    next_cp_id[0] += 1
    merchant_pool_v2[norm] = mid
    merchant_alias_v2[norm] = mid
    if domain and domain not in merchant_alias_v2:
        merchant_alias_v2[domain] = mid
    for alias in generate_merchant_aliases(raw, norm):
        if alias not in merchant_alias_v2:
            merchant_alias_v2[alias] = mid
    cp_info[mid] = {'name': norm, 'type': 'merchant'}
    return (mid, 'new')


# --- Run full pipeline ---
sorted_txns2 = txns.sort_values('date').reset_index(drop=True)
pipeline2_results = []

for idx, row in sorted_txns2.iterrows():
    entity_type = row['entity_type']
    raw = row['raw_merchant']
    cleaned = row['cleaned']

    if entity_type == 'empty' or not raw.strip():
        pipeline2_results.append({
            'idx': idx, 'raw': raw, 'entity_type': 'empty',
            'cp_id': None, 'method': 'empty', 'bank': row['bank'],
        })
        continue

    if entity_type == 'person':
        canonical, aliases = normalize_person(raw)
        cp_id, method = resolve_person_cp(raw, canonical, aliases)
        pipeline2_results.append({
            'idx': idx, 'raw': raw, 'entity_type': 'person',
            'cp_id': cp_id, 'method': method, 'bank': row['bank'],
        })
    elif entity_type == 'merchant':
        cp_id, method = resolve_merchant_cp(raw, cleaned)
        pipeline2_results.append({
            'idx': idx, 'raw': raw, 'entity_type': 'merchant',
            'cp_id': cp_id, 'method': method, 'bank': row['bank'],
        })
    else:  # unknown — default to merchant pipeline
        cp_id, method = resolve_merchant_cp(raw, cleaned)
        pipeline2_results.append({
            'idx': idx, 'raw': raw, 'entity_type': 'unknown',
            'cp_id': cp_id, 'method': method, 'bank': row['bank'],
        })

p2_df = pd.DataFrame(pipeline2_results)

# ============================================================
# REPORTS
# ============================================================
print('=' * 60)
print('FULL PIPELINE WITH PERSON/MERCHANT SPLIT')
print('=' * 60)

# Entity type distribution
print('\n=== Entity Type Distribution ===')
for t in ['merchant', 'person', 'unknown', 'empty']:
    count = (p2_df.entity_type == t).sum()
    print(f'  {t:10s}: {count:5d} ({count/len(p2_df)*100:.1f}%)')

# Match rate per type
print('\n=== Match Rate per Entity Type ===')
for t in ['merchant', 'person', 'unknown']:
    subset = p2_df[p2_df.entity_type == t]
    matched = subset[~subset.method.isin(['new', 'empty'])]
    total = len(subset)
    print(f'  {t:10s}: {len(matched)}/{total} matched '
          f'({len(matched)/max(total,1)*100:.1f}%)')

# Resolution methods
print('\n=== Resolution Methods ===')
def method_cat_v2(m: str) -> str:
    if m == 'exact_alias': return 'Exact alias'
    elif m.startswith('domain'): return 'Domain extraction'
    elif m.startswith('fuzzy_token'): return 'Fuzzy (token_set)'
    elif m.startswith('fuzzy_partial'): return 'Fuzzy (partial)'
    elif m.startswith('fuzzy_person'): return 'Fuzzy (person)'
    elif m == 'new': return 'New counterparty'
    elif m == 'empty': return 'Unresolvable'
    return m

p2_df['method_cat'] = p2_df.method.apply(method_cat_v2)
for cat, count in p2_df.method_cat.value_counts().items():
    print(f'  {cat:25s}: {count:5d} ({count/len(p2_df)*100:.1f}%)')

# Overall stats
resolved = p2_df[p2_df.method != 'empty']
matched = resolved[resolved.method != 'new']

print(f'\n=== Overall ===')
n_persons = sum(1 for v in cp_info.values() if v['type'] == 'person')
n_merchants = sum(1 for v in cp_info.values() if v['type'] == 'merchant')
print(f'Total counterparties: {len(cp_info)}')
print(f'  Persons:   {n_persons}')
print(f'  Merchants: {n_merchants}')
print(f'Overall match rate: {len(matched)}/{len(resolved)} = '
      f'{len(matched)/len(resolved)*100:.1f}%')

# Cross-source person matches
print('\n=== Cross-Source Person Matches ===')
person_rows = p2_df[p2_df.entity_type == 'person']
cross_source_persons = 0
for cp_id, info in cp_info.items():
    if info['type'] == 'person':
        banks = person_rows[person_rows.cp_id == cp_id]['bank'].unique()
        if len(banks) > 1:
            raws = person_rows[person_rows.cp_id == cp_id]['raw'].unique()
            cross_source_persons += 1
            print(f'  "{info["name"]}" (id={cp_id}) in {list(banks)}: '
                  f'{list(raws)[:4]}')
print(f'Total cross-source persons: {cross_source_persons}')

# 20 random samples
print('\n=== 20 Random Sample Review ===')
sample = p2_df[p2_df.method != 'empty'].sample(20, random_state=42)
for _, row in sample.iterrows():
    cp_name = cp_info.get(row['cp_id'], {}).get('name', '?')
    print(f'  [{row.entity_type:8s}] [{row.method:20s}] '
          f'"{row.raw}" -> "{cp_name}"')

# All fuzzy matches
print('\n=== All Fuzzy Matches ===')
fuzzy = p2_df[p2_df.method.str.startswith('fuzzy')].drop_duplicates(
    ['raw', 'cp_id']
)
for _, row in fuzzy.iterrows():
    cp_name = cp_info.get(row['cp_id'], {}).get('name', '?')
    print(f'  [{row.entity_type:8s}] [{row.method:20s}] '
          f'"{row.raw}" -> "{cp_name}"')

# Missed merges analysis
print('\n=== Missed Merges Analysis ===')
for cp_type in ['merchant', 'person']:
    type_names = [
        (info['name'], cid)
        for cid, info in cp_info.items()
        if info['type'] == cp_type
    ]
    name_list = [n for n, _ in type_names]
    checked: set[tuple[str, str]] = set()
    missed = []
    for name in name_list:
        if len(name) < 3:
            continue
        results = process.extract(
            name, name_list,
            scorer=fuzz.token_set_ratio, limit=3, score_cutoff=75
        )
        for match, score, _ in results:
            if match != name and (match, name) not in checked:
                missed.append((name, match, int(score)))
                checked.add((name, match))
                checked.add((match, name))

    if missed:
        missed.sort(key=lambda x: -x[2])
        print(f'\n  Potential missed {cp_type} merges:')
        for a, b, score in missed[:15]:
            print(f'    {score}% "{a}" <-> "{b}"')